## Sequence Classification

In [2]:
from huggingface_hub import notebook_login

notebook_login()

### Load the dataset

In [20]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")
imdb["test"][0]

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

### Preprocess

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [5]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [23]:
tokenized_imdb = imdb.map(preprocess_function, batched=True)

In [8]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Evaluate

In [10]:
import evaluate

accuracy = evaluate.load("accuracy")

import numpy as np

def compute_metrics(eval_pred):
    predictions, labels= eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

### Train

In [11]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [16]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id)

small_train = tokenized_imdb["train"].shuffle(seed=42).select(range(1000))
small_eval = tokenized_imdb["test"].shuffle(seed=42).select(range(300))

training_args = TrainingArguments(
    output_dir="my_awesome_imdb_beat-model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    max_steps=50,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="no",
    load_best_model_at_end=False,
    push_to_hub=False,
    logging_steps=10,
    dataloader_pin_memory=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss,Accuracy
25,0.691261,0.659503,0.810600
50,0.619482,0.596954,0.839640


TrainOutput(global_step=50, training_loss=0.6630046653747559, metrics={'train_runtime': 493.055, 'train_samples_per_second': 1.623, 'train_steps_per_second': 0.101, 'total_flos': 105518562241920.0, 'train_loss': 0.6630046653747559, 'epoch': 0.03198976327575176})

In [17]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/hedonwang/my_awesome_imdb_beat-model/commit/ee701f77ed650def1cb5199ad9f1672942368b14', commit_message='End of training', commit_description='', oid='ee701f77ed650def1cb5199ad9f1672942368b14', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hedonwang/my_awesome_imdb_beat-model', endpoint='https://huggingface.co', repo_type='model', repo_id='hedonwang/my_awesome_imdb_beat-model'), pr_revision=None, pr_num=None)

In [18]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="my_awesome_imdb_beat-model")
classifier("I've been waiting for a Hugging Face course my whole life.")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.5114730596542358}]